In [1]:
!pip install numpy pandas

  Using cached pandas-3.0.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 20.4 MB/s eta 0:00:0000:0100:01
Using cached pandas-3.0.6-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.8 MB)


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

INTERIM_DIR = Path("../../resources/interim/Skyteam_Timetable")
PROCESSED_DIR = Path("../../resources/processed")

RAW_FILE = INTERIM_DIR / "doc_0_0.csv"
MERGED_DEBUG_FILE = INTERIM_DIR / "doc_merged.csv"
FINAL_FILE = PROCESSED_DIR / "doc_final.csv"

SPLIT_POINT = 9
MIN_COLS = 12

STOP_WORDS = ("operated", "time", "travel")
JUNK_KEYWORDS = ("validity", "days", "dep", "arr", "time", "flight", "aircraft", "travel")

def read_raw_lines(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        lines = pd.Series(f.readlines())

    lines = lines.str.replace("\ufeff", "", regex=False).str.rstrip("\r\n")
    lines = lines[lines.str.len() > 0].reset_index(drop=True)
    return lines


def fix_shifted_pipes(lines: pd.Series):
    is_candidate = (
        lines.str.startswith("||")
        & (lines.str.len() > 2)
        & (lines.str[2] != "|")
    )

    fixed = lines.copy()
    subset = lines[is_candidate]
    if subset.empty:
        return fixed

    core = subset.str.rstrip("|")
    right_pipes = subset.str.len() - core.str.len()
    needs_fix = right_pipes > 0

    subset_fixed = subset[needs_fix]
    core_fixed = core[needs_fix]
    pipes_fixed = right_pipes[needs_fix]

    new_values = "||" + pipes_fixed.apply(lambda n: "|" * n) + core_fixed.str[2:]
    fixed.loc[new_values.index] = new_values
    return fixed

def split_left_right(lines: pd.Series):
    raw_split = lines.str.split("|", n=SPLIT_POINT, expand=True)

    left_df = raw_split.iloc[:, :SPLIT_POINT].fillna("")

    if raw_split.shape[1] > SPLIT_POINT:
        remainder = raw_split[SPLIT_POINT].fillna("")
    else:
        remainder = pd.Series([""] * len(lines), index=lines.index)

    right_df = remainder.str.split("|", expand=True).fillna("")

    assert len(left_df) == len(lines), "left-часть потеряла/добавила строки"
    assert len(right_df) == len(lines), "right-часть потеряла/добавила строки"

    return left_df.reset_index(drop=True), right_df.reset_index(drop=True)

def shift_cells_left(df: pd.DataFrame):
    arr = df.to_numpy(dtype=object)
    is_empty = (arr == "") | pd.isna(arr)
    order = np.argsort(is_empty, axis=1, kind="stable")
    shifted = np.take_along_axis(arr, order, axis=1)
    return pd.DataFrame(shifted, columns=df.columns, index=df.index).fillna("")

def row_text(df: pd.DataFrame):
    return df.astype(str).agg("|".join, axis=1)


def is_junk_or_empty(text: pd.Series):
    clean = (
        text.str.replace("\ufeff", "", regex=False)
            .str.replace('"', "", regex=False)
            .str.strip()
    )
    is_blank = clean.str.replace("|", "", regex=False).str.strip().eq("")

    pattern = "|".join(JUNK_KEYWORDS)
    has_keyword = clean.str.lower().str.contains(pattern, regex=True)

    return is_blank | has_keyword


def keep_if_second_column_meaningful(df: pd.DataFrame, text: pd.Series):
    has_stopword = text.str.lower().str.contains("|".join(STOP_WORDS), regex=True)

    if df.shape[1] > 1:
        col2 = df.iloc[:, 1].astype(str).str.strip().str.lower()
        col2_ok = ~col2.isin(["", "nan", "none", "null"])
    else:
        col2_ok = pd.Series(False, index=df.index)

    return has_stopword | col2_ok


def clean_segment(df: pd.DataFrame):
    df = df.apply(lambda col: col.astype(str).str.strip())
    df = shift_cells_left(df)

    text = row_text(df)
    mask = ~is_junk_or_empty(text)
    df, text = df[mask].reset_index(drop=True), text[mask].reset_index(drop=True)

    mask2 = keep_if_second_column_meaningful(df, text)
    df = df[mask2].reset_index(drop=True)

    return df


def merge_segments(left_df: pd.DataFrame, right_df: pd.DataFrame, min_cols: int = MIN_COLS):
    total_cols = max(left_df.shape[1], right_df.shape[1], min_cols)

    def pad(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        for i in range(df.shape[1], total_cols):
            df[i] = ""
        df.columns = range(total_cols)
        return df

    return pd.concat([pad(left_df), pad(right_df)], axis=0, ignore_index=True)

def propagate_flight_context(df: pd.DataFrame):
    df = df.copy()
    col1 = df[0].astype(str)
    col1_upper = col1.str.upper()

    is_from = col1_upper.str.contains("FROM:")
    is_to = col1_upper.str.contains("TO:")
    is_operated = col1_upper.str.contains("OPERATED BY:")

    def parse_after_colon(s: pd.Series) -> pd.Series:
        return s.str.split(":", n=1).str[1].fillna("").str.strip(' "')

    col2 = df[1].astype(str) if df.shape[1] > 1 else pd.Series("", index=df.index)
    col3 = df[2].astype(str) if df.shape[1] > 2 else pd.Series("", index=df.index)
    parsed = parse_after_colon(col1)

    from_val = pd.Series(np.where(is_from, np.where(col2 != "", col2, parsed), np.nan), index=df.index)
    from_abb = pd.Series(np.where(is_from, col3, np.nan), index=df.index)
    to_val = pd.Series(np.where(is_to, np.where(col2 != "", col2, parsed), np.nan), index=df.index)
    to_abb = pd.Series(np.where(is_to, col3, np.nan), index=df.index)

    df[7] = from_val.ffill().fillna("")
    df[8] = from_abb.ffill().fillna("")
    df[9] = to_val.ffill().fillna("")
    df[10] = to_abb.ffill().fillna("")

    carrier = pd.Series(np.where(is_operated, parsed, np.nan), index=df.index)
    carrier_for_prev_row = carrier.shift(-1)
    df[11] = np.where(carrier_for_prev_row.notna(), carrier_for_prev_row, df[11])

    # строки-маркеры больше не нужны — их данные уже "протянуты" куда надо
    df = df[~(is_from | is_to | is_operated)].reset_index(drop=True)
    return df

def validate_final(df: pd.DataFrame, n_cols: int = 11):
    check = df.iloc[:, :n_cols].replace("", np.nan)
    bad_mask = check.isna().any(axis=1)
    bad_rows = df[bad_mask]

    if bad_rows.empty:
        return

def build_pipeline(save_debug: bool = True):
    INTERIM_DIR.mkdir(parents=True, exist_ok=True)
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

    lines = read_raw_lines(RAW_FILE)
    lines = fix_shifted_pipes(lines)

    left_raw, right_raw = split_left_right(lines)

    left_clean = clean_segment(left_raw)
    right_clean = clean_segment(right_raw)

    merged = merge_segments(left_clean, right_clean)
    
    if save_debug:
        merged.to_csv(MERGED_DEBUG_FILE, sep="|", index=False, header=False, encoding="utf-8")

    processed = propagate_flight_context(merged)

    processed.columns = ['VALIDITY', 'DAYS', 'DEP_TIME', 'ARR_TIME', 'FLIGHT', 'AIRCRAFT', 'TRAVEL_TIME', 'FROM_CITY', 'FROM_CODE', 'TO_CITY', 'TO_CODE', 'OPERATED_BY']
    processed.to_csv(FINAL_FILE, sep="|", index=False, encoding="utf-8")

    validate_final(processed)
    return processed


if __name__ == "__main__":
    build_pipeline(save_debug=False)